# ShapleyFlow SHAP Value Debugging

This notebook traces through ShapleyFlow computation for a single instance to identify why SHAP values appear inflated by ~100× compared to other methods.

**Observation:**
- Other methods (AsymmetricShapley, CausalShapley): SHAP values range [-0.2, 0.15]
- ShapleyFlow: SHAP values range [-15, 15]
- Y range: [-4.38, 3.35]

**Investigation:**
1. Load a single test instance
2. Compute ShapleyFlow step-by-step with detailed logging
3. Check edge attributions, node attributions, and final SHAP values
4. Verify efficiency axiom: sum(SHAP) ≈ f(x) - f(baseline)
5. Compare to other methods

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import json

# Add parent directory to path
sys.path.append('/Users/juanrios/Documents/master_thesis')

from predictive_models.predictive_models import LGBMRegressor
from explainability_models import (
    ShapleyFlowWrapper, ShapleyFromScratch, AsymmetricShapley
)

# Directories
BASE_DIR = Path('/Users/juanrios/Documents/master_thesis')
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
CAUSAL_DIR = BASE_DIR / 'data' / 'causal'
EXPLAINABILITY_DIR = BASE_DIR / 'data' / 'explainability'
MODELS_DIR = BASE_DIR / 'models'

print("✅ Imports successful")

✅ Imports successful


## 1. Load Data and Model

In [2]:
# Dataset configuration
dataset = 'mixed_conf_f50_s1000_p50'
model_name = 'lgbm'
discovery_method = 'lingam'  # or 'lingam'

# Load train/test data
train_path = PROCESSED_DIR / f"{dataset}_train.parquet"
test_path = PROCESSED_DIR / f"{dataset}_test.parquet"

train_data = pd.read_parquet(train_path)
test_data = pd.read_parquet(test_path)

X_train = train_data.drop(columns=['Y'])
y_train = train_data['Y']
X_test = test_data.drop(columns=['Y'])
y_test = test_data['Y']

# Load model
model_path = MODELS_DIR / f"{dataset}_{model_name}"
model = LGBMRegressor.load(str(model_path))

print(f"Dataset: {dataset}")
print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Y range: [{y_train.min():.2f}, {y_train.max():.2f}]")
print(f"Y test range: [{y_test.min():.2f}, {y_test.max():.2f}]")

LightGBM model loaded from /Users/juanrios/Documents/master_thesis/models/mixed_conf_f50_s1000_p50_lgbm.pkl
Dataset: mixed_conf_f50_s1000_p50
Train shape: (800, 50)
Test shape: (200, 50)
Y range: [-3.99, 6.35]
Y test range: [-5.86, 6.35]


## 2. Select a Single Test Instance

In [3]:
# Select first test instance
instance_idx = 0
test_instance = X_test.iloc[[instance_idx]]
true_y = y_test.iloc[instance_idx]

# Get model prediction
pred_y = model.predict(test_instance)[0]

print(f"Instance {instance_idx}:")
print(f"  True Y: {true_y:.4f}")
print(f"  Predicted Y: {pred_y:.4f}")
print(f"  Feature values (first 10): {test_instance.values[0][:10]}")

Instance 0:
  True Y: 0.6456
  Predicted Y: -0.4112
  Feature values (first 10): [ 0.33185122 -0.30629677  0.19821468 -0.09891843 -0.43697709  1.17312498
  0.34614028  0.83053809  1.4656966   0.01209883]


## 3. Load Causal Graph

In [4]:
# Load PC or LiNGAM results
results_path = CAUSAL_DIR / f"{dataset}_{discovery_method}_results.json"
with open(results_path, 'r') as f:
    discovery_results = json.load(f)

causal_graph = np.array(discovery_results['adjacency_matrix'])
feature_names = discovery_results['feature_names']

# Count edges
n_edges = np.sum(causal_graph != 0)
print(f"Causal discovery method: {discovery_method.upper()}")
print(f"Graph shape: {causal_graph.shape}")
print(f"Number of edges: {n_edges}")
print(f"Edges to Y: {np.sum(causal_graph[:, -1] != 0)}")

Causal discovery method: LINGAM
Graph shape: (51, 51)
Number of edges: 275
Edges to Y: 7


## 4. Compute ShapleyFlow with Detailed Logging

In [5]:
# Prepare data for ShapleyFlow (needs Y column)
background_data_flow = X_train.sample(n=100, random_state=42).copy()
background_data_flow['Y'] = y_train.loc[background_data_flow.index]

test_instance_flow = test_instance.copy()
test_instance_flow['Y'] = true_y

print(f"Background data shape: {background_data_flow.shape}")
print(f"Test instance shape: {test_instance_flow.shape}")

Background data shape: (100, 51)
Test instance shape: (1, 51)


In [6]:
# Create ShapleyFlow explainer with VERY FEW samples for debugging
flow_explainer = ShapleyFlowWrapper(
    model=model,
    background_data=background_data_flow,
    causal_graph=causal_graph,
    y_index=len(feature_names) - 1,  # Y is last column
    n_samples=5,  # Only 5 trials for debugging
    random_state=42,
    use_path_sampling=True,
    paths_per_source=10  # Only 10 paths per source
)

print("ShapleyFlow explainer created")
print(f"  Sources: {flow_explainer.source_nodes}")
print(f"  Graph edges: {sum(len(v) for v in flow_explainer.graph_structure.values())}")

ShapleyFlow explainer created
  Sources: [22, 35, 44]
  Graph edges: 275


In [7]:
# Compute SHAP values
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

flow_values = flow_explainer.explain(test_instance_flow)

print(f"\n{'='*80}")
print(f"ShapleyFlow SHAP values computed")
print(f"{'='*80}")
print(f"Shape: {flow_values.shape}")
print(f"Sum: {flow_values.sum():.4f}")
print(f"Min: {flow_values.min():.4f}")
print(f"Max: {flow_values.max():.4f}")


ShapleyFlow SHAP values computed
Shape: (1, 50)
Sum: -0.1902
Min: -0.1557
Max: 0.0017


## 5. Check Efficiency Axiom

The efficiency axiom states: **sum(SHAP values) ≈ f(x) - f(baseline)**

In [8]:
# Compute baseline prediction (mean of background)
baseline_pred = model.predict(background_data_flow.drop(columns=['Y']).sample(n=20, random_state=42)).mean()

# Expected sum
expected_sum = pred_y - baseline_pred

# Actual sum
actual_sum = flow_values.sum()

print(f"Efficiency Axiom Check:")
print(f"  f(x) = {pred_y:.4f}")
print(f"  f(baseline) ≈ {baseline_pred:.4f}")
print(f"  Expected sum: {expected_sum:.4f}")
print(f"  Actual sum: {actual_sum:.4f}")
print(f"  Ratio: {actual_sum / expected_sum if expected_sum != 0 else 'inf':.2f}×")
print(f"\n⚠️ If ratio >> 1, SHAP values are inflated!")

Efficiency Axiom Check:
  f(x) = -0.4112
  f(baseline) ≈ 0.2154
  Expected sum: -0.6266
  Actual sum: -0.1902
  Ratio: 0.30×

⚠️ If ratio >> 1, SHAP values are inflated!


## 6. Compare to Other Methods

In [9]:
# Compute ShapleyFromScratch
background_scratch = X_train.sample(n=100, random_state=42)

scratch_explainer = ShapleyFromScratch(
    model,
    background_scratch,
    n_samples=100,
    random_state=42
)

scratch_values = scratch_explainer.explain(test_instance, method='monte_carlo')

print(f"ShapleyFromScratch SHAP values:")
print(f"  Sum: {scratch_values.sum():.4f}")
print(f"  Min: {scratch_values.min():.4f}")
print(f"  Max: {scratch_values.max():.4f}")

Computing Shapley values from scratch using monte_carlo method...
ShapleyFromScratch SHAP values:
  Sum: -0.5369
  Min: -0.2279
  Max: 0.1313


In [12]:
# Compute AsymmetricShapley
import json
results_path = f"data/causal/{dataset}_{discovery_method}_results.json"
with open(results_path, 'r') as f:
    results_json = json.load(f)
adjacency_matrix = np.array(results_json['adjacency_matrix'])

asymmetric_explainer = AsymmetricShapley(
    model,
    background_scratch,
    causal_graph=adjacency_matrix,
    n_samples=100,
    random_state=42
)

asymmetric_values = asymmetric_explainer.explain(test_instance, method='monte_carlo')

print(f"\nAsymmetricShapley SHAP values:")
print(f"  Sum: {asymmetric_values.sum():.4f}")
print(f"  Min: {asymmetric_values.min():.4f}")
print(f"  Max: {asymmetric_values.max():.4f}")

Computing Path-based Asymmetric Shapley values using monte_carlo method...
 Progress: 1/1 instances

AsymmetricShapley SHAP values:
  Sum: -0.4081
  Min: -0.1148
  Max: 0.1338


## 7. Compare Top Features Across Methods

In [13]:
# Get top 10 features by absolute SHAP value for each method
def get_top_features(shap_values, feature_names, n=10):
    abs_values = np.abs(shap_values.flatten())
    top_indices = np.argsort(abs_values)[-n:][::-1]
    return [(feature_names[i], shap_values.flatten()[i]) for i in top_indices]

flow_top = get_top_features(flow_values, X_test.columns.tolist())
scratch_top = get_top_features(scratch_values, X_test.columns.tolist())
asymmetric_top = get_top_features(asymmetric_values, X_test.columns.tolist())

print("Top 10 Features Comparison:\n")
print(f"{'Feature':<10} {'ShapFlow':<15} {'Scratch':<15} {'Asymmetric':<15}")
print("=" * 60)

for i in range(10):
    feat_flow, val_flow = flow_top[i]
    feat_scratch, val_scratch = scratch_top[i]
    feat_asym, val_asym = asymmetric_top[i]
    
    print(f"{feat_flow:<10} {val_flow:>14.4f} {val_scratch:>14.4f} {val_asym:>14.4f}")

Top 10 Features Comparison:

Feature    ShapFlow        Scratch         Asymmetric     
X44               -0.1557        -0.2279         0.1338
X26               -0.0158         0.1313        -0.1148
X35               -0.0139        -0.1094        -0.1106
X38               -0.0090         0.1043         0.1071
X22                0.0017        -0.1032        -0.1037
X41                0.0015        -0.0929        -0.0941
X16                0.0007        -0.0386        -0.0403
X21                0.0004        -0.0374        -0.0341
X3                 0.0000        -0.0337        -0.0337
X13                0.0000        -0.0324        -0.0328


## 8. Investigate Edge vs Node Attributions

ShapleyFlow computes **edge** attributions first, then aggregates to **node** attributions. Let's check if the issue is in aggregation.

In [14]:
# Access the last ShapleyFlow object used (from inside wrapper)
# We'll need to recompute with access to internal state

from explainability_models.shapley_values import ShapleyFlow

# Manually create ShapleyFlow for single instance
x_foreground = {j: test_instance_flow.values[0][j] for j in range(len(feature_names))}

# Sample random background
bg_idx = 42
x_background = {j: background_data_flow.values[bg_idx, j] for j in range(len(feature_names))}

flow = ShapleyFlow(
    graph_structure=flow_explainer.graph_structure,
    background_data=background_data_flow.values,
    model=model,
    source_nodes=flow_explainer.source_nodes,
    sink_node=flow_explainer.y_index,
    n_samples=5,
    random_state=42,
    feature_names=feature_names,
    use_path_sampling=True,
    paths_per_source=10
)

# Compute edge attributions
edge_attrs = flow.compute(x_foreground, x_background)

print(f"Edge Attributions:")
print(f"  Number of edges: {len(edge_attrs)}")
print(f"  Sum of edge attributions: {sum(edge_attrs.values()):.4f}")
print(f"  Min edge attribution: {min(edge_attrs.values()):.4f}")
print(f"  Max edge attribution: {max(edge_attrs.values()):.4f}")

Edge Attributions:
  Number of edges: 275
  Sum of edge attributions: -0.0374
  Min edge attribution: -0.0209
  Max edge attribution: 0.0107


In [15]:
# Get node attributions
node_attrs = flow.get_node_attributions()

print(f"\nNode Attributions:")
print(f"  Number of nodes with attributions: {len(node_attrs)}")
print(f"  Sum of node attributions: {sum(node_attrs.values()):.4f}")
print(f"  Min node attribution: {min(node_attrs.values()):.4f}")
print(f"  Max node attribution: {max(node_attrs.values()):.4f}")

# Show top 10 nodes
sorted_nodes = sorted(node_attrs.items(), key=lambda x: abs(x[1]), reverse=True)[:10]
print(f"\nTop 10 Nodes by Absolute Attribution:")
for node_idx, attr in sorted_nodes:
    if node_idx < len(feature_names) - 1:  # Exclude Y
        print(f"  {feature_names[node_idx]}: {attr:.4f}")


Node Attributions:
  Number of nodes with attributions: 48
  Sum of node attributions: -0.0374
  Min node attribution: -0.0247
  Max node attribution: 0.0198

Top 10 Nodes by Absolute Attribution:
  X35: -0.0247
  X26: -0.0209
  X44: 0.0198
  X38: -0.0140
  X21: 0.0018
  X41: 0.0016
  X16: -0.0009
  X0: 0.0000
  X1: 0.0000
  X2: 0.0000


## 9. Diagnostic Summary

Based on the analysis above, we should be able to identify:
1. Whether the inflation happens at the edge level or node aggregation
2. Whether the efficiency axiom is violated (actual_sum >> expected_sum)
3. How ShapleyFlow compares to other methods in magnitude and ranking

**Next Steps:**
- If edge attributions are correct but node aggregations are inflated → issue in `get_node_attributions()`
- If edge attributions are already inflated → issue in path sampling normalization
- If efficiency axiom is severely violated → fundamental computation error

In [16]:
# Final diagnostic summary
print("="*80)
print("DIAGNOSTIC SUMMARY")
print("="*80)

print(f"\n1. SHAP Value Magnitudes:")
print(f"   ShapleyFlow:       [{flow_values.min():.2f}, {flow_values.max():.2f}], sum={flow_values.sum():.2f}")
print(f"   ShapleyFromScratch: [{scratch_values.min():.2f}, {scratch_values.max():.2f}], sum={scratch_values.sum():.2f}")
print(f"   AsymmetricShapley:  [{asymmetric_values.min():.2f}, {asymmetric_values.max():.2f}], sum={asymmetric_values.sum():.2f}")

print(f"\n2. Efficiency Axiom:")
print(f"   Expected sum (f(x) - f(baseline)): {expected_sum:.2f}")
print(f"   ShapleyFlow ratio: {flow_values.sum() / expected_sum if expected_sum != 0 else 'inf':.2f}×")
print(f"   Scratch ratio: {scratch_values.sum() / expected_sum if expected_sum != 0 else 'inf':.2f}×")

print(f"\n3. Edge vs Node Attribution Check:")
print(f"   Sum of edge attributions: {sum(edge_attrs.values()):.2f}")
print(f"   Sum of node attributions: {sum(node_attrs.values()):.2f}")
print(f"   Ratio (node/edge): {sum(node_attrs.values()) / sum(edge_attrs.values()) if sum(edge_attrs.values()) != 0 else 'inf':.2f}×")

if abs(flow_values.sum() / expected_sum) > 4 if expected_sum != 0 else False:
    print(f"\n⚠️ WARNING: ShapleyFlow SHAP values are inflated by ~{flow_values.sum() / expected_sum:.0f}×")
    print("   Possible causes:")
    print("   - Path sampling normalization issue")
    print("   - Edge aggregation counting edges multiple times")
    print("   - Missing division during node attribution computation")
else:
    print(f"\n✅ ShapleyFlow appears correctly calibrated")

DIAGNOSTIC SUMMARY

1. SHAP Value Magnitudes:
   ShapleyFlow:       [-0.16, 0.00], sum=-0.19
   ShapleyFromScratch: [-0.23, 0.13], sum=-0.54
   AsymmetricShapley:  [-0.11, 0.13], sum=-0.41

2. Efficiency Axiom:
   Expected sum (f(x) - f(baseline)): -0.63
   ShapleyFlow ratio: 0.30×
   Scratch ratio: 0.86×

3. Edge vs Node Attribution Check:
   Sum of edge attributions: -0.04
   Sum of node attributions: -0.04
   Ratio (node/edge): 1.00×

✅ ShapleyFlow appears correctly calibrated
